# 🕵️ Dune Detective
## Trends in wind, soil moisture, rainfall, heat and vegetation to fight sand encroachment
**NASA Space Apps Challenge 2026 — "Be an Earth System Trend Detective"**
**Author: Ouiam Ouatat — DATA part of the team**

---
### 🎯 Why this notebook exists
Our project fights **sand encroachment and dune movement**, first around **Merzouga (Morocco)**, then in other African regions. Our main solution is **planting vegetation that fixes sand**, combined with other measures (windbreak fences, mulching, water harvesting).

Before saying *where* and *when* to plant, we must understand **how the local climate is changing**. The challenge asks exactly this: find **what changes, where, by how much, and whether the change is statistically significant**.

### 🧩 The 4 questions of the challenge and our answer
| Challenge question | Our answer in this notebook |
|---|---|
| **What** changes? | Wind, soil moisture, rainfall, surface/soil temperature (and vegetation if Earth Engine is ready). |
| **Where**? | Merzouga (pilot site) + 3 other African sites to compare and prepare generalisation. |
| **By how much**? | A **slope per decade** (for example: +0.05 m/s per decade). |
| **Is it significant**? | A statistical test (**Mann-Kendall**). If p < 0.05 the trend is considered real; otherwise we honestly say it is not demonstrated. |

### 📖 How to read each step
- **What I do** — the action of the cell.
- **Why** — the scientific or technical reason.
- **What it brings to the team** — who uses the results.
- **What I produce** — the files created.

### ✅ Before you start
1. A **NASA Earthdata** account (urs.earthdata.nasa.gov) with the application **"NASA GESDISC DATA ARCHIVE"** authorised (once only: Applications → Authorized Apps → Approve More Applications).
2. Run the cells **in order** with Shift+Enter. Do not use "Run all" (some cells wait for an action from you).
3. Big NASA files are stored on the **Colab disk**, never on your computer.

### 📚 Mini-glossary (so the whole team speaks the same language)
- **NASA Earthdata**: NASA's portal to its data (free account).
- **MERRA-2**: a NASA "reanalysis". It rebuilds past weather consistently (wind, temperature...) on a global grid. Cells of about **50 km**, since 1980.
- **GLDAS**: a NASA land-surface model (soil moisture, rain, soil temperature...). Cells of about **25 km**, since 2000.
- **NetCDF (.nc, .nc4)**: the scientific file format. One file holds several variables on a grid over time.
- **U10M / V10M**: wind at 10 m, east-west (U) and north-south (V) components, in m/s. Wind speed is computed from them.
- **NDVI**: vegetation index from -1 to 1. Near 0 = bare soil or sand; above 0.3 = vegetation.
- **BSI**: Bare Soil Index. The higher it is, the more bare the surface. Used here as a *change proxy* only.
- **Sen's slope**: the typical change per year (here shown per decade), robust to extreme values.
- **Mann-Kendall test / p-value**: tells whether a trend is real or due to chance. **p < 0.05 = significant.**
- **Anomaly**: a value compared with what is normal for the same calendar month.
- **Correlation**: how much two variables move together. It is **not** proof of cause.

### 🔗 The thread of the investigation and the files produced
| Variable | Its role in the sand story | Source (step) |
|---|---|---|
| **Wind** | It **transports** the sand. | MERRA-2 (step 4) |
| **Soil moisture** | Dry soil = mobile sand; moist soil = plants survive. | GLDAS (step 5) |
| **Rainfall** | It **brings water**. Extreme rain can cause flash floods (and can be harvested). | GLDAS (step 5), NASA POWER (step 6b) |
| **Surface / soil temperature** | Heat dries the soil and stresses plants. | MERRA-2 (step 4), GLDAS (step 5), MODIS (step 6, optional) |
| **NDVI / BSI (vegetation / bare soil)** | What we **protect** and how the land **responds**. | MODIS via Earth Engine (step 6, optional), team BSI file (step 6b) |

| Step | What I do | File produced |
|---|---|---|
| 1-3 | Install, configure, connect to NASA | — |
| 4 | Wind and surface temperature (MERRA-2) | — |
| 5 | Soil moisture, rain, soil temperature (GLDAS) | — |
| 6 | NDVI and satellite heat (optional) | — |
| 6b | Team data: NASA POWER rain + MODIS BSI | `team_bsi_vs_rain.png` |
| 7 | Compute trends | `trend_results.csv`, `annual_series.csv` |
| 8 / 8b | Summary chart / links between variables | `trend_summary.png`, `variable_links.png` |
| 9 | Interactive dashboard | `trend_dashboard.html` |
| 10 | Results in plain sentences | — |
| 11 | Proof file | `README.md`, `data_provenance.csv` |
| 12-13 | Share and merge with the team | Google Drive folder, zip |

**All variables go through the same method** (annual mean, slope, significance test), so they are comparable and fit in the same dashboard.

## Step 1 — Install the tools
**What I do**: install the Python libraries that Colab does not have by default (`earthaccess`, `pymannkendall`, `xarray`, `netCDF4`, `h5netcdf`, `dask`, `plotly`, `tabulate`).

**Why**: without them the next cells fail.
**Good to know**: a red message about dependency conflicts (`fsspec`...) is a **harmless warning** for this notebook.

In [ ]:
!pip install -q earthaccess pymannkendall xarray netCDF4 h5netcdf dask plotly tabulate

## Step 2 — My settings: sites, periods, output folder
**What I do**: choose **where** and **over which years** I study the data. **This is the only cell to edit to change location.**

**Why**:
- **Merzouga** is the pilot site and **3 other sites** (Timbuktu, Nouakchott, Siwa) allow comparison. The challenge reminds us that the same variable can change in one direction in one region and the opposite way in another.
- Both periods are 2000-2025 so that wind, soil and rain can be compared over the same years. You can set `START_WIND = 1990` for a longer wind history (more files, slower).
- Each site value is the **average of a small box** around the point (`BOX_HALF_WIDTH` degrees). This avoids empty values for coastal sites (ocean cells have no land data).

**What it brings to the team**: to **generalise** to another African region, add one line to `SITES` (name, latitude, longitude). Coordinates are approximate: adjust if needed.

In [ ]:
import os, shutil, glob, datetime
import numpy as np, pandas as pd, xarray as xr, dask
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pymannkendall as mk

# --- My study sites: name -> (latitude, longitude) ---
SITES = {
    "Merzouga":   (31.10, -4.01),    # pilot site (Morocco)
    "Timbuktu":   (16.77, -3.01),    # Sahel (Mali)
    "Nouakchott": (18.10, -15.50),   # Saharan coast, moved slightly inland (Mauritania)
    "Siwa":       (29.20, 25.52),    # Libyan desert (Egypt)
}

# --- My study periods (complete years) ---
START_WIND, END_WIND = 2000, 2025    # MERRA-2 monthly (exists since 1980)
START_SOIL, END_SOIL = 2000, 2025    # GLDAS 2.1 (exists since 2000)
BOX_HALF_WIDTH = 0.3                 # degrees around each site (averaged)

# --- My folders ---
OUT = "outputs_trends"               # small files I will share
os.makedirs(OUT, exist_ok=True)
os.makedirs("data", exist_ok=True)   # big NASA files stay on the Colab disk

# --- Two "boxes" filled along the notebook ---
annual = {}        # for each variable: a table (years x sites) of annual means
provenance = []    # list of my data sources (my proof)
print("Study sites:", list(SITES))

## Step 3 — Connect to NASA and prepare my 3 functions
### 3a. Login
**What I do**: log in to my NASA Earthdata account. A box asks for my username and password.
**Why**: NASA data is free but protected by an account. My password never appears in the notebook, so I can share it safely.

**If an `EulaNotAccepted` error appears later**: the application "NASA GESDISC DATA ARCHIVE" was not authorised in the account (Applications → Authorized Apps → Approve More Applications).

In [ ]:
import earthaccess
auth = earthaccess.login(strategy="interactive", persist=True)
print("Login successful:", auth.authenticated)

### 3b. The 3 working functions
1. **`download`**: finds the files of a NASA product (short name + version) and downloads them to the Colab disk.
2. **`extract`**: opens all files, keeps only the chosen variables and **averages a small box around each site** (ignoring empty ocean cells). It prints the **unit** of each variable: always read it (for example Kelvin instead of °C).
3. **`annualize`**: computes the **annual mean**, keeping only **complete years** (12 months).

**Why the annual mean**: seasons (hot summer, cold winter) would hide the trend. Comparing year against year shows the underlying change.
**What it brings to the team**: for any **new data**, just change the short name, version and variable list.

In [ ]:
def download(short_name, version, start, end, folder):
    """Search the NASA files of a product and download them to the Colab disk."""
    results = earthaccess.search_data(short_name=short_name, version=version,
                                      temporal=(f"{start}-01-01", f"{end}-12-31"))
    print(f"{short_name}: {len(results)} files found")
    return earthaccess.download(results, folder)

def extract(files, variables, sites, half=BOX_HALF_WIDTH):
    """Open all files, keep my variables, average a small box around each site (one single read)."""
    def keep(d):
        return d[[v for v in variables if v in d]]
    ds = xr.open_mfdataset(files, combine="by_coords", preprocess=keep)
    for v in variables:
        if v in ds:
            print(f"  {v} | unit: {ds[v].attrs.get('units')} | {ds[v].attrs.get('long_name')}")
        else:
            print(f"  {v} | NOT FOUND in the files")
    boxes = {}
    for name, (lat, lon) in sites.items():
        b = ds.sel(lat=slice(lat - half, lat + half), lon=slice(lon - half, lon + half))
        if b.sizes["lat"] == 0 or b.sizes["lon"] == 0:          # box too small for the grid -> nearest cell
            b = ds.sel(lat=[lat], lon=[lon], method="nearest")
        boxes[name] = b.mean(dim=["lat", "lon"], skipna=True)
    computed = dask.compute(boxes)[0]                           # one single pass over all files
    return {name: d.to_dataframe() for name, d in computed.items()}

def annualize(series):
    """Annual mean, keeping only complete years (12 months)."""
    by_year = series.groupby(series.index.year)
    mean = by_year.mean()
    return mean[by_year.count() == 12]

## Step 4 — Wind and surface temperature (MERRA-2 monthly)
**What I do**: download the **monthly** MERRA-2 files (product `M2TMNXSLV`, version `5.12.4`) and extract:
- **wind at 10 m** (`U10M`, `V10M`) for each site;
- **surface skin temperature** (`TS`) — over the desert this is essentially the **temperature of the sand surface**;
- the wind **variances** (`Var_U10M`, `Var_V10M`), if present.

**Why wind**: it transports the sand. **Why temperature**: heat dries the soil and stresses plants.

**⚠️ Important scientific point**: the monthly mean of U and V **underestimates** the real wind because opposite directions cancel out. When variances exist, I compute a "quadratic" speed that is closer to the real wind. I do **not** compare these values with the sand-motion threshold: that needs hourly data (see the Meteomatics idea in the team plan).

**What I produce**: annual wind and surface temperature for each site. **Duration**: about 5-15 minutes. I read the lines showing the **units**.

In [ ]:
# 1) Download the monthly MERRA-2 files (on the Colab disk).
wind_files = download("M2TMNXSLV", "5.12.4", START_WIND, END_WIND, "data/merra2_monthly")

# 2) Extract my variables for each site.
wind_vars = ["U10M", "V10M", "Var_U10M", "Var_V10M", "TS"]
wind = extract(wind_files, wind_vars, SITES)

# 3) Wind speed for each site.
speeds = {}
for name, table in wind.items():
    mean_speed = np.sqrt(table.U10M**2 + table.V10M**2)
    if "Var_U10M" in table and "Var_V10M" in table:
        speed = np.sqrt(table.U10M**2 + table.V10M**2 + table.Var_U10M + table.Var_V10M)
        wind_method = "quadratic speed (with variances)"
    else:
        speed = mean_speed
        wind_method = "speed of the monthly mean wind (underestimates real wind)"
    speeds[name] = annualize(speed)
annual["Wind at 10 m (m/s)"] = pd.DataFrame(speeds)
print("Wind method used:", wind_method)

# 4) Surface skin temperature (Kelvin in the file: subtract 273.15 to get °C).
if all("TS" in t for t in wind.values()):
    annual["Surface skin temperature (°C, MERRA-2)"] = pd.DataFrame(
        {name: annualize(t["TS"] - 273.15) for name, t in wind.items()})
else:
    print("Variable TS is missing: surface temperature skipped.")

# 5) Source note (my proof): the DOI is read directly from the file.
attrs = xr.open_dataset(wind_files[0]).attrs
provenance.append({"product": "MERRA-2 M2TMNXSLV v5.12.4", "variables": "U10M, V10M, TS (+ variances if present)",
                   "period": f"{START_WIND}-{END_WIND}", "files": len(wind_files),
                   "doi": attrs.get("identifier_product_doi", "see the product page"), "note": wind_method})
annual["Wind at 10 m (m/s)"].tail()   # last years, to check

## Step 5 — Soil moisture, rainfall and soil temperature (GLDAS monthly)
**What I do**: download the monthly GLDAS Noah files (product `GLDAS_NOAH025_M`, version `2.1`) and extract:
- **soil moisture 0-10 cm** (`SoilMoi0_10cm_inst`);
- **root-zone moisture** (`RootMoist_inst`);
- **rainfall rate** (`Rainf_f_tavg`), converted from kg/m²/s to **mm/day** (× 86,400);
- **soil temperature 0-10 cm** (`SoilTMP0_10cm_inst`), converted from Kelvin to °C.

**Why**: dry soil leaves sand mobile and prevents plants from surviving; rain brings the water needed for planting; hot sand kills young plants.
**What it brings to the team**: whoever studies **plant species** gets the water and heat conditions of each site.
**Limit**: GLDAS is a **model** (25 km cells), not a field measurement.

In [ ]:
# 1) Download the monthly GLDAS files.
soil_files = download("GLDAS_NOAH025_M", "2.1", START_SOIL, END_SOIL, "data/gldas_monthly")

# 2) Extract my variables for each site.
soil_vars = ["SoilMoi0_10cm_inst", "RootMoist_inst", "Rainf_f_tavg", "SoilTMP0_10cm_inst"]
soil = extract(soil_files, soil_vars, SITES)

def has(var):
    return all(var in t for t in soil.values())

# 3) Surface soil moisture (0-10 cm).
annual["Soil moisture 0-10 cm (kg/m²)"] = pd.DataFrame({n: annualize(t["SoilMoi0_10cm_inst"]) for n, t in soil.items()})

# 4) Root-zone moisture, if available.
if has("RootMoist_inst"):
    annual["Root-zone moisture (kg/m²)"] = pd.DataFrame({n: annualize(t["RootMoist_inst"]) for n, t in soil.items()})

# 5) Rainfall: kg/m²/s -> mm/day (1 kg/m² of water = 1 mm; 86,400 seconds per day).
if has("Rainf_f_tavg"):
    annual["Rainfall (mm/day, GLDAS)"] = pd.DataFrame({n: annualize(t["Rainf_f_tavg"] * 86400) for n, t in soil.items()})
else:
    print("Rainf_f_tavg is missing: rainfall skipped.")

# 6) Soil temperature 0-10 cm (Kelvin -> °C), if available.
if has("SoilTMP0_10cm_inst"):
    annual["Soil temperature 0-10 cm (°C, GLDAS)"] = pd.DataFrame({n: annualize(t["SoilTMP0_10cm_inst"] - 273.15) for n, t in soil.items()})
else:
    print("SoilTMP0_10cm_inst is missing: soil temperature skipped.")

# 7) Source note.
attrs = xr.open_dataset(soil_files[0]).attrs
provenance.append({"product": "GLDAS Noah 0.25° monthly v2.1", "variables": ", ".join(soil_vars),
                   "period": f"{START_SOIL}-{END_SOIL}", "files": len(soil_files),
                   "doi": attrs.get("identifier_product_doi", "see the product page"), "note": "land-surface model, ~25 km cells"})
annual["Soil moisture 0-10 cm (kg/m²)"].tail()

## Step 6 — Vegetation (NDVI) and satellite heat — OPTIONAL
**What I do**: with **Google Earth Engine**, for each site from 2001 to 2024 I compute:
- the **annual maximum NDVI** (MODIS satellite): the amount of vegetation;
- the **daytime land surface temperature** (MODIS satellite): a satellite measurement, more direct than the model.

**Why NDVI is central to our project**: it shows **what we protect** (oases, crops), **where the sand is bare** (where planting makes sense), whether **vegetation is receding or advancing**, and it lets us **measure the result of a planting** (NDVI should rise).

**How to enable it**: you need a Google Earth Engine account and a project (free for non-commercial use). Set `USE_GEE = True` and replace `MY_PROJECT` with your project ID. **While it is not ready, leave `False` and the rest of the notebook works.**

**Honesty note**: this cell could not be tested before delivery. If it fails, copy the error message.

In [ ]:
USE_GEE = False             # <-- set to True when your Earth Engine project is ready
EE_PROJECT = "MY_PROJECT"   # <-- write your Earth Engine project ID here

if USE_GEE:
    import ee
    ee.Authenticate(); ee.Initialize(project=EE_PROJECT)

    # --- NDVI: yearly maximum, averaged over a 10 km disc around the site ---
    nd = ee.ImageCollection("MODIS/061/MOD13Q1").select("NDVI")
    ndvi_sites = {}
    for name, (lat, lon) in SITES.items():
        zone = ee.Geometry.Point([lon, lat]).buffer(10000)
        def per_year(y):
            y = ee.Number(y)
            image = nd.filterDate(ee.Date.fromYMD(y, 1, 1), ee.Date.fromYMD(y, 12, 31)).max()
            m = image.reduceRegion(ee.Reducer.mean(), zone, 250, maxPixels=1e9).get("NDVI")
            return ee.Feature(None, {"year": y, "ndvi": m})
        fc = ee.FeatureCollection(ee.List.sequence(2001, 2024).map(per_year)).getInfo()
        # factor 0.0001: MODIS NDVI values are stored multiplied by 10,000
        ndvi_sites[name] = pd.Series({int(f["properties"]["year"]): (f["properties"].get("ndvi") or np.nan) * 0.0001
                                      for f in fc["features"]})
    annual["Maximum annual NDVI (0 to 1)"] = pd.DataFrame(ndvi_sites)

    # --- Daytime land surface temperature (satellite): annual mean ---
    lt = ee.ImageCollection("MODIS/061/MOD11A2").select("LST_Day_1km")
    lst_sites = {}
    for name, (lat, lon) in SITES.items():
        zone = ee.Geometry.Point([lon, lat]).buffer(10000)
        def per_year2(y):
            y = ee.Number(y)
            image = lt.filterDate(ee.Date.fromYMD(y, 1, 1), ee.Date.fromYMD(y, 12, 31)).mean()
            m = image.reduceRegion(ee.Reducer.mean(), zone, 1000, maxPixels=1e9).get("LST_Day_1km")
            return ee.Feature(None, {"year": y, "lst": m})
        fc = ee.FeatureCollection(ee.List.sequence(2001, 2024).map(per_year2)).getInfo()
        # factor 0.02 then -273.15: values stored as Kelvin x 50 -> °C
        lst_sites[name] = pd.Series({int(f["properties"]["year"]): (f["properties"].get("lst") or np.nan) * 0.02 - 273.15
                                     for f in fc["features"]})
    annual["Daytime land surface temperature (°C, MODIS satellite)"] = pd.DataFrame(lst_sites)

    provenance.append({"product": "MODIS MOD13Q1 and MOD11A2 v061 (via Google Earth Engine)", "variables": "annual max NDVI, daytime LST",
                       "period": "2001-2024", "files": "-", "doi": "see the product pages", "note": "250 m (NDVI), 1 km (LST), 10 km disc"})
else:
    print("NDVI skipped (USE_GEE = False). This is normal if Earth Engine is not ready.")

## Step 6b — Team data: NASA POWER rainfall and MODIS Bare Soil Index (BSI)
**What I do**: I load the three files shared by the team and put them in the same method as everything else.
1. `power_precipitation_daily.csv` — daily rain (NASA POWER). I compute **annual rainfall** and the **number of extreme-rain days per year**.
2. `power_monthly_rain_anomalies.csv` — monthly rain compared with the normal for the same month.
3. `modis_sand_bsi_monthly.csv` — monthly **BSI** (MODIS): higher = more bare surface.

**How**: drag the three CSV files into the Colab **Files** panel inside a folder named `team_data` (create it if needed), then run the cell. If the files are missing, the cell simply skips.

**Why**: extreme-rain days are the rare pulses of water we could **harvest for planting**; the BSI is a satellite signal of how the **land surface responds** after wet periods.

**⚠️ Before trusting it**: (1) the files do not say which **latitude/longitude** they cover; set `TEAM_SITE` once the team confirms. (2) The BSI is a *change proxy only* and has just one image per month, with gaps. (3) POWER is a coarse gridded product: very large one-day values must be checked before being quoted. (4) Correlations with ~50 points are fragile — a lead, not a proof.

In [ ]:
TEAM_DIR = "team_data"
TEAM_SITE = "Team site"      # <-- rename once the team confirms the location of these files
os.makedirs(TEAM_DIR, exist_ok=True)
f_daily = f"{TEAM_DIR}/power_precipitation_daily.csv"
f_month = f"{TEAM_DIR}/power_monthly_rain_anomalies.csv"
f_bsi   = f"{TEAM_DIR}/modis_sand_bsi_monthly.csv"

if os.path.exists(f_daily):
    rd = pd.read_csv(f_daily, parse_dates=["date"])
    g = rd.groupby(rd.date.dt.year)
    complete = g.rain_mm_day.count() >= 360                     # keep complete years only
    annual["Annual rainfall (mm/year, NASA POWER)"] = pd.DataFrame({TEAM_SITE: g.rain_mm_day.sum()[complete]})
    annual["Extreme rain days per year (days, NASA POWER)"] = pd.DataFrame({TEAM_SITE: g.extreme_rain_day.sum()[complete].astype(float)})
    print("Mean annual rainfall: %.0f mm" % annual["Annual rainfall (mm/year, NASA POWER)"][TEAM_SITE].mean())
    print("Wettest days on record:")
    print(rd.nlargest(5, "rain_mm_day")[["date", "rain_mm_day"]].round(1).to_string(index=False))
    provenance.append({"product": "NASA POWER daily precipitation (team file)", "variables": "rain_mm_day, extreme_rain_day",
                       "period": f"{rd.date.dt.year.min()}-{rd.date.dt.year.max()}", "files": 1,
                       "doi": "see NASA POWER", "note": "location to be confirmed by the team; coarse gridded product"})
else:
    print("power_precipitation_daily.csv not found in team_data/ : skipped.")

if os.path.exists(f_bsi) and os.path.exists(f_month):
    bsi = pd.read_csv(f_bsi, parse_dates=["date"])
    rm = pd.read_csv(f_month, parse_dates=["date"])
    r = mk.original_test(bsi.bsi_mean.values)
    print(f"\nBSI monthly trend test: {r.trend}, p = {r.p:.4f}  (n = {len(bsi)} monthly points)")
    print("Caution: monthly points are strongly linked to each other, so this p-value is optimistic.")

    # --- Does the BSI change after wet months? (lagged correlation, a lead and not a proof) ---
    bsi["ym"] = bsi.date.dt.to_period("M"); rm["ym"] = rm.date.dt.to_period("M")
    idx = pd.period_range(min(bsi.ym.min(), rm.ym.min()), max(bsi.ym.max(), rm.ym.max()), freq="M")
    b = bsi.set_index("ym").bsi_mean.reindex(idx)
    a = rm.set_index("ym").anomaly_mm.reindex(idx)
    print("\nCorrelation between rain anomaly (month t) and BSI (month t + lag):")
    for lag in range(0, 4):
        pair = pd.concat([a.shift(lag), b], axis=1).dropna()
        print(f"  lag {lag} month(s): r = {pair.corr().iloc[0, 1]:+.2f}   (n = {len(pair)})")

    # --- Figure: rain anomalies on top, BSI below, extreme-rain days as red lines ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    ax1.bar(rm.date, rm.anomaly_mm, width=20, color=np.where(rm.anomaly_mm >= 0, "#1f77b4", "#d98c3a"))
    ax1.axhline(0, color="black", lw=0.8); ax1.set_ylabel("Rain anomaly (mm vs normal)")
    ax1.set_title("Wet and dry months vs the Bare Soil Index (BSI) — BSI is a change proxy only")
    ax2.plot(bsi.date, bsi.bsi_mean, marker="o"); ax2.set_ylabel("BSI (higher = more bare)")
    if os.path.exists(f_daily):
        for d in rd[rd.extreme_rain_day & (rd.date >= bsi.date.min())].date:
            for ax in (ax1, ax2):
                ax.axvline(d, color="red", alpha=0.5, lw=1)
        ax2.text(0.01, 0.04, "red lines = extreme-rain days", transform=ax2.transAxes, color="red")
    for ax in (ax1, ax2): ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/team_bsi_vs_rain.png", dpi=150); plt.show()
    provenance.append({"product": "MODIS BSI monthly (team file) + NASA POWER monthly anomalies", "variables": "bsi_mean, anomaly_mm",
                       "period": f"{bsi.date.dt.year.min()}-{bsi.date.dt.year.max()}", "files": 2,
                       "doi": "see MODIS / NASA POWER", "note": "BSI = change proxy only; one image per month with gaps"})
else:
    print("BSI or monthly rain file not found in team_data/ : skipped.")

## Step 7 — Compute the trends: what, where, by how much, significant?
**What I do**: for **each variable and each site**, I apply two tools to the series of annual means:
- **Sen's slope**: the typical change **per year**, multiplied by 10 to get a change **per decade** (and as a % of the mean when relevant);
- the **Mann-Kendall test**: it gives a **p-value**. If **p < 0.05** the trend is considered **significant** (little chance it is due to luck).

**Why these tools**: they are **robust** to extreme values and assume no particular distribution. They are the standard for climate series.

**How I read the result**:
- `significant = True`: the trend is demonstrated over this period.
- `significant = False`: **I cannot claim** there is a trend. This is **not** proof that nothing changes, especially with 25-35 years.

**What I produce**: `trend_results.csv` and `annual_series.csv`. Series with fewer than 8 years are skipped.

In [ ]:
def trend(series):
    """Sen's slope + Mann-Kendall test on a series of annual means."""
    s = series.dropna()
    r = mk.original_test(s.values)
    return {"n_years": len(s), "mean": s.mean(),
            "slope_year": r.slope, "intercept": r.intercept,                 # to draw the trend line
            "slope_per_decade": r.slope * 10,
            "pct_per_decade": r.slope * 10 / s.mean() * 100 if s.mean() != 0 else np.nan,
            "p_value": r.p, "significant": bool(r.p < 0.05), "trend": r.trend}

tend, rows = {}, []
for variable, table in annual.items():
    for site in table.columns:
        if table[site].dropna().shape[0] < 8:
            print("Series too short or empty, skipped:", variable, "|", site); continue
        res = trend(table[site])
        tend[(variable, site)] = res
        rows.append({"variable": variable, "site": site,
                     **{k: res[k] for k in ["n_years", "mean", "slope_per_decade", "pct_per_decade", "p_value", "significant", "trend"]}})

results = pd.DataFrame(rows).round(4)
results.to_csv(f"{OUT}/trend_results.csv", index=False)                                  # all results
pd.concat({v: d for v, d in annual.items()}, axis=1).to_csv(f"{OUT}/annual_series.csv")  # annual series
results   # display the table

## Step 8 — Summary chart: where and by how much it changes
**What I do**: for each variable I plot the **slope per decade** at each site.
- **Dark** bar: **significant** trend (p < 0.05). **Light** bar: **not demonstrated**.
- The **p** value is written above each bar.

**Why**: at a glance we see **where** it changes (which site), **in which direction** (above or below zero) and **whether it is reliable** (colour). This is the image that best answers the 4 challenge questions.
**What I produce**: `trend_summary.png`, ready for the presentation.

In [ ]:
variables = list(dict.fromkeys(results.variable))
ncols = 3; nrows = int(np.ceil(len(variables) / ncols))
figure, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.4 * nrows), squeeze=False)
for ax, variable in zip(axes.flat, variables):
    d = results[results.variable == variable]
    colors = ["#c0392b" if s else "#d5d8dc" for s in d.significant]        # dark = significant
    ax.bar(d.site, d.slope_per_decade, color=colors, edgecolor="black")
    for i, (_, row) in enumerate(d.iterrows()):                           # p-value on each bar
        ax.text(i, row.slope_per_decade, f"p={row.p_value:.2f}", ha="center",
                va="bottom" if row.slope_per_decade >= 0 else "top", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(variable, fontsize=9)
    ax.set_ylabel("Slope per decade (variable unit)", fontsize=8)
    ax.tick_params(axis="x", rotation=25); ax.grid(alpha=0.3)
for ax in list(axes.flat)[len(variables):]:
    ax.axis("off")
figure.suptitle("Trends by site — dark = significant (p < 0.05), light = not demonstrated", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT}/trend_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 8b — Links between variables (for one site)
**What I do**: for the focus site, I compute the **correlation** between all variables from year to year and show it as a colour map.
- **+1**: both go up and down together. **-1**: when one goes up the other goes down. **0**: no visible link.

**Why**: the challenge stresses the **interconnected** Earth system. For example: are the hottest years also the driest?

**⚠️ Caution**: a correlation **is not a cause**, and with 25 years it is fragile. I present it as a **lead**, not as proof.
**What I produce**: `variable_links.png`.

In [ ]:
SITE_FOCUS = "Merzouga"    # I can change the site here

# Keep only years common to all variables.
links = pd.DataFrame({v: d[SITE_FOCUS] for v, d in annual.items() if SITE_FOCUS in d}).dropna()

if links.shape[1] >= 2 and len(links) >= 8:
    corr = links.corr()
    labels = [c[:28] + ("…" if len(c) > 28 else "") for c in corr.columns]
    figure, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(image, label="Correlation")
    ax.set_title(f"Links between variables — {SITE_FOCUS} ({len(links)} common years)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/variable_links.png", dpi=150)
    plt.show()
else:
    print("Not enough variables or common years to compute correlations.")

## Step 9 — The interactive dashboard
**What I do**: build an interactive chart with a **dropdown menu** "variable — site". For each choice it shows the **annual curve**, the **trend line** (dashed) and a **verdict sentence** (increase, decrease, or no demonstrated trend, with slope and p-value).

**Why**: the whole team can explore the results **without coding**. The file is standalone: it opens in any browser (double-click). Any variable added above appears automatically in the menu.

**What I produce**: `trend_dashboard.html` (download it from the Colab Files panel, then double-click).

In [ ]:
def verdict(r, variable=""):
    """The sentence shown above the chart."""
    if r["significant"]:
        pct = "" if "°C" in variable else f" ({r['pct_per_decade']:+.1f} %)"
        direction = "Significant increase" if r["slope_per_decade"] > 0 else "Significant decrease"
        return f"{direction}: {r['slope_per_decade']:+.3f} per decade{pct}, p = {r['p_value']:.3f}"
    return f"No significant trend (slope {r['slope_per_decade']:+.3f} per decade, p = {r['p_value']:.2f})"

figure = go.Figure()
choices = []     # list of (variable, site, verdict)
for (variable, site), r in tend.items():
    s = annual[variable][site].dropna()
    figure.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines+markers", name="Annual value", visible=False))
    figure.add_trace(go.Scatter(x=s.index, y=r["intercept"] + r["slope_year"] * np.arange(len(s)),
                                mode="lines", name="Trend (Sen's slope)", line=dict(dash="dash", width=3), visible=False))
    choices.append((variable, site, verdict(r, variable)))

def visible(k):
    """For choice number k, show only its 2 curves."""
    mask = [False] * (2 * len(choices)); mask[2 * k] = mask[2 * k + 1] = True
    return mask

buttons = [dict(label=f"{v} — {s}", method="update",
                args=[{"visible": visible(k)},
                      {"title": f"{v} — {s}",
                       "annotations": [dict(text=txt, x=0.5, y=1.08, xref="paper", yref="paper", showarrow=False)]}])
           for k, (v, s, txt) in enumerate(choices)]

for curve in figure.data[:2]:
    curve.visible = True
figure.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=1.25, xanchor="left")],
                     title=f"{choices[0][0]} — {choices[0][1]}", xaxis_title="Year", yaxis_title="Value",
                     annotations=[dict(text=choices[0][2], x=0.5, y=1.08, xref="paper", yref="paper", showarrow=False)],
                     margin=dict(t=140))
figure.write_html(f"{OUT}/trend_dashboard.html")   # standalone file to share
figure.show()

## Step 10 — Reading my results as sentences
**What I do**: I turn the table into **clear sentences**, variable by variable. ✔ = **significant** trend; • = **no demonstrated trend**; ⚠ = sites go in **opposite directions** for the same variable.

**Why**: this is the text I can **say aloud** in front of the team or the jury.
**Reminder**: "no demonstrated trend" is a **valid result**. A trend **does not prove by itself** that sand encroachment increases.

In [ ]:
for variable, d in results.groupby("variable", sort=False):
    print(f"\n=== {variable} ===")
    for row in d.itertuples():
        pct = "" if "°C" in variable else f" ({row.pct_per_decade:+.1f} %)"
        if row.significant:
            word = "increases" if row.slope_per_decade > 0 else "decreases"
            print(f"✔ At {row.site}, the value {word} significantly: {row.slope_per_decade:+.3f} per decade{pct} (p = {row.p_value:.3f}).")
        else:
            print(f"• At {row.site}: no demonstrated trend (slope {row.slope_per_decade:+.3f} per decade, p = {row.p_value:.2f}).")
    sig = d[d.significant]
    if (sig.slope_per_decade > 0).any() and (sig.slope_per_decade < 0).any():
        print("⚠ Sites do not all go the same way: the trend depends on the region.")

## Step 11 — My proof file (README)
**What I do**: generate a `README.md` that says **where the data comes from** (product, version, period, number of files, DOI), **which variables** were analysed, **the method**, **the significant trends found** and **the limits**.
**Why**: it is my **proof of work** and the instruction sheet for the team and the jury. It shows that nothing is invented: everything comes from traceable data.
**What I produce**: `README.md` and `data_provenance.csv`.

In [ ]:
prov = pd.DataFrame(provenance)
sig = results[results.significant]
sig_lines = "\n".join(
    f"- {r.variable} at {r.site}: {r.slope_per_decade:+.3f} per decade" + ("" if "°C" in r.variable else f" ({r.pct_per_decade:+.1f} %)") + f", p={r.p_value:.3f}"
    for r in sig.itertuples()) or "- No significant trend detected in the series studied."
try:
    prov_text = prov.to_markdown(index=False)
except Exception:
    prov_text = prov.to_string(index=False)

readme = f"""# Dune Detective — trend results
Generated on {datetime.date.today()} in Google Colab from NASA data (access through earthaccess, Earthdata account) and team files.
Author: Ouiam Ouatat (DATA part) — NASA Space Apps 2026, challenge "Be an Earth System Trend Detective".

## Study sites
{", ".join(f"{n} ({la:.2f}, {lo:.2f})" for n, (la, lo) in SITES.items())}

## Variables analysed
{chr(10).join("- " + v for v in annual)}

## Data sources
{prov_text}

## Method
Annual means (complete years only), Sen's slope, Mann-Kendall test (threshold p < 0.05).

## Significant trends found
{sig_lines}

## Limits
- MERRA-2 (~50 km) and GLDAS (~25 km) cells describe the regional climate, not precise planting points.
- Series of 25 years: a test can miss a real weak trend.
- Monthly mean wind underestimates the real wind (see the method used in the provenance table).
- A trend is not a cause: it does not prove by itself an increase in sand encroachment.
- The BSI is a change proxy only; team-file locations must be confirmed.

## Files
- trend_results.csv: all results (what, where, by how much, significant)
- annual_series.csv: the annual series
- trend_summary.png: summary chart
- variable_links.png: correlations between variables (one site)
- team_bsi_vs_rain.png: wet/dry months vs bare soil index (if team files were loaded)
- trend_dashboard.html: interactive dashboard (open in a browser)
"""
open(f"{OUT}/README.md", "w", encoding="utf-8").write(readme)
prov.to_csv(f"{OUT}/data_provenance.csv", index=False)
print(readme)

## Step 12 — Give my results to my teammates
**What I do**: export **only the small result files** (not the big NASA files).

**Option A (recommended) — Google Drive**: copy the `outputs_trends` folder to my Google Drive, then in Drive right-click the folder → **Share** and add my teammates' emails. Teammates must **download** the `.html` to open it (Drive only shows its code).
**Option B — zip file**: one single file to download and send by message.

| File | For whom | To do what |
|---|---|---|
| `trend_dashboard.html` | Everyone | Explore the results (download, then double-click) |
| `trend_results.csv` | Analysts / presenters | Reference numbers |
| `annual_series.csv` | Anyone who wants to redo a chart | Raw annual data |
| `trend_summary.png`, `variable_links.png`, `team_bsi_vs_rain.png` | Presentation | Ready-made images |
| `README.md`, `data_provenance.csv` | Jury, team | Proof of where the data comes from |

In [ ]:
# Option A: copy to my Google Drive (a window asks for authorisation).
from google.colab import drive
drive.mount("/content/drive")
target = "/content/drive/MyDrive/NASA_SpaceApps_Dune_Detective"
shutil.copytree(OUT, target, dirs_exist_ok=True)
print("Results copied to my Google Drive:", target)

In [ ]:
# Option B: a zip to download (the download starts in my browser).
shutil.make_archive("trend_results", "zip", OUT)
from google.colab import files
files.download("trend_results.zip")

## Step 13 — Merge the results of the whole team (optional)
**What I do**: when teammates analyse other variables, I ask them to save their results with **the same columns** as `trend_results.csv`, in a file named `trend_NAME.csv`, in the shared Drive folder. This cell **assembles** them into one table.
**Why**: one method for everyone = comparable results and one single dashboard at the end.

In [ ]:
COLUMNS = ["variable", "site", "n_years", "mean", "slope_per_decade", "pct_per_decade", "p_value", "significant", "trend"]

def merge(folder):
    """Assemble all trend_*.csv files of the folder into one table."""
    files_ = [f for f in sorted(glob.glob(f"{folder}/**/trend_*.csv", recursive=True))
              if not f.endswith("trend_all_variables.csv") and not f.endswith("trend_results.csv")]
    parts = []
    for f in files_:
        d = pd.read_csv(f)
        missing = [c for c in COLUMNS if c not in d.columns]
        if missing:
            print("Ignored (missing columns:", missing, "):", f); continue
        d["source_file"] = os.path.basename(f)
        parts.append(d[COLUMNS + ["source_file"]])
    if not parts:
        print("No valid file found."); return None
    everything = pd.concat(parts, ignore_index=True)
    everything.to_csv(f"{folder}/trend_all_variables.csv", index=False)
    print(len(parts), "files merged ->", len(everything), "rows")
    return everything

# merge(target)   # <- uncomment when teammates have put their CSV files in the Drive folder

## Guide for my teammates

### What I did
I accessed NASA data (Earthdata account), extracted wind, surface temperature, soil moisture, rainfall and soil temperature for 4 African sites, loaded the team's NASA POWER rain and MODIS BSI files, and computed for each one **what changes, by how much, and whether it is significant**. I produced a dashboard, charts, tables and a proof file.

### How to use my results
- **Explore**: download `trend_dashboard.html`, double-click it, choose "variable — site".
- **Present**: use `trend_summary.png`, `team_bsi_vs_rain.png` and the sentences of step 10.
- **Reuse the numbers**: `trend_results.csv` and `annual_series.csv`.
- **Check the sources**: `README.md`.

### What the results do NOT say (say it clearly)
- NASA cells are **25-50 km**: they describe the **regional climate**, not exact planting spots.
- A trend **is not a cause**: it does not prove by itself that sand encroachment increases.
- "No demonstrated trend" means we cannot claim it, not that nothing changes.

### Common errors and fixes
| Message | Meaning | Fix |
|---|---|---|
| `EulaNotAccepted` | The GESDISC application is not authorised | Earthdata → Applications → Authorized Apps → Approve More Applications |
| `Series too short or empty, skipped` | A site has no land data | Move the site inland or increase `BOX_HALF_WIDTH` |
| Cell runs for a very long time | Reading hundreds of NASA files | Wait (10-25 min), keep the tab open, or shorten the period |
| Session disconnected | Colab lost the downloaded files | Re-run from step 1; files will download again |
| `NDVI skipped` | Earth Engine not enabled | Normal. Set `USE_GEE = True` only when the project is ready |

### Next steps
1. Enable vegetation (NDVI) with Google Earth Engine (step 6).
2. Place plantations with a precise map (Landsat/Sentinel, 10-30 m) crossing vegetation, bare sand and wind direction.
3. Validate in the field before any planting.
4. Choose species with the water and heat conditions found here.
5. Study flash floods with daily rain data (see step 6b) and match them with water-harvesting techniques (zaï pits, half-moons, sand dams).